# ShelfWise — Full Test Harness

**Run every cell top to bottom (Kernel -> Restart & Run All). No manual setup, no extra files, no data to add — everything this notebook needs is already committed in this repo (the seed CSVs under `data/datasets/`, the pinned dependency lists, the whole `src/` tree).**

What this notebook proves, section by section:

1. The environment installs cleanly from this repo alone.
2. Static checks (lint) pass.
3. The full backend test suite passes (currently 280+ tests: every agent, the HITL approval loop, connectors, security, decision science, multimodal, the product-identity catalog).
4. The golden-scenario eval gate passes (the deterministic proof that the full cascade — scan -> inventory -> expiry -> demand -> opportunity -> simulation -> critic -> executive -> HITL — produces the right evidence, the right numbers, a visible Critic rejection, and a visible learning moment).
5. The FastAPI app answers real requests in-process (every major surface: health, the golden cascade, HITL approve, the product-identity catalog, connectors, observability).
6. A real ASGI server (uvicorn) boots on a real port and answers real HTTP requests — not just the in-process test client.
7. Optional: a real inference call through the AMD MI300X/vLLM (or Fireworks) endpoint, **only if you set `LLM_BASE_URL` / `LLM_API_KEY` first** — this is the one thing that needs something *outside* the repo, because it's a live network credential, not a file.

Every check below is wrapped so one failure does not stop the rest of the notebook — you always get to the final summary table at the bottom, which is the one cell worth reading first.

## 0. Setup — repo path, install, versions

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"repo root: {ROOT}")
print(f"src import path present: {SRC.exists()}")
print(f"python: {sys.version}")

# ---- shared result tracking so ONE failing section never stops the notebook ----
RESULTS: list[dict] = []


def record(name: str, ok: bool, detail: str = "") -> None:
    RESULTS.append({"section": name, "ok": ok, "detail": detail})
    banner = "PASS" if ok else "FAIL"
    print(f"\n[{banner}] {name}" + (f"  - {detail}" if detail else ""))


def run(name: str, fn) -> None:
    """Run one check; catch anything so the rest of the notebook still executes."""
    try:
        fn()
        record(name, True)
    except AssertionError as exc:
        record(name, False, f"assertion failed: {exc}")
    except Exception as exc:  # noqa: BLE001 - deliberately broad, this is a test harness
        record(name, False, f"{type(exc).__name__}: {exc}")

### Install dependencies (runtime + dev) from this repo's own `pyproject.toml`

Nothing is downloaded from anywhere except PyPI for the packages themselves — no repo assets are fetched from the network.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[dev]"],
    capture_output=True,
    text=True,
)
print(install.stdout[-2000:])
if install.returncode != 0:
    print(install.stderr[-2000:])
record("pip install -e .[dev]", install.returncode == 0, f"exit={install.returncode}")

## 1. Static checks — lint (`ruff`)

In [ ]:
def _lint() -> None:
    result = subprocess.run(
        [sys.executable, "-m", "ruff", "check", "src", "tests", "scripts"],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
    assert result.returncode == 0, "ruff reported lint errors (see output above)"


run("Lint (ruff)", _lint)

## 2. Full backend test suite

Every agent, the HITL approval/reject lifecycle, connectors (Shopify/Lightspeed/Square/SAP/Odoo/Syspro), the security gateway, tenant isolation, decision science, multimodal, inference, and the product-identity catalog.

In [ ]:
def _pytest() -> None:
    env = dict(os.environ)
    env["PYTHONPATH"] = str(SRC)
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=ROOT,
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
    assert result.returncode == 0, "pytest reported failures (see output above)"


run("Full test suite (pytest)", _pytest)

## 3. Golden-scenario eval harness

The deterministic proof-of-safety gate: it runs the whole `scan -> inventory -> expiry -> demand -> opportunity -> simulation -> critic -> executive -> HITL` cascade against the seeded scenario and checks every invariant — grounded evidence, the exact simulation numbers, a visible Critic rejection, a visible learning moment, and a token/cost ceiling.

In [ ]:
def _eval_gate() -> None:
    env = dict(os.environ)
    env["PYTHONPATH"] = str(SRC)
    result = subprocess.run(
        [sys.executable, "-m", "shelfwise_eval"],
        cwd=ROOT,
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
    assert result.returncode == 0, "eval gate reported a failing check (see output above)"


run("Golden-scenario eval gate", _eval_gate)

## 4. In-process API smoke

Drives the real FastAPI app object directly (`TestClient` — the same mechanism the test suite uses, so this is not a separate reimplementation): health, the full golden cascade, HITL approval, the product-identity catalog (the identity-resolution proof), connectors, and the observability snapshot.

In [ ]:
def _api_smoke() -> None:
    from fastapi.testclient import TestClient
    from shelfwise_backend.app import app

    client = TestClient(app)

    health = client.get("/health")
    assert health.status_code == 200, health.text
    print("GET /health ->", health.json())

    readiness = client.get("/readiness")
    assert readiness.status_code == 200 and readiness.json()["ready"] is True, readiness.text
    print("GET /readiness -> ready=True")

    golden = client.get("/demo/golden")
    assert golden.status_code == 200, golden.text
    decision = golden.json()["decision"]
    agents = [item["agent"] for item in golden.json()["evidence"]]
    assert decision["status"] == "pending", decision
    print("GET /demo/golden -> agents:", agents)
    print("  decision action:", decision["action"]["type"], "| status:", decision["status"])

    approve = client.post(f"/decisions/{decision['id']}/approve")
    assert approve.status_code == 200, approve.text
    approved = approve.json()["decision"]
    assert approved["status"] == "approved", approved
    print("POST /decisions/{id}/approve -> status:", approved["status"])
    print("  recovered:", approved["outcome"]["rand_recovered"])

    rejection = client.get("/demo/critic-rejection")
    assert rejection.status_code == 200, rejection.text
    assert rejection.json()["decision"]["status"] == "rejected"
    print("GET /demo/critic-rejection -> visible Critic rejection confirmed")

    product = client.post(
        "/catalog/products", json={"product_id": "nb_milk", "name": "Full Cream Milk"}
    )
    variant = client.post(
        "/catalog/products/nb_milk/variants", json={"variant_id": "nb_milk_1l"}
    )
    client.post(
        "/catalog/identifiers",
        json={"variant_id": "nb_milk_1l", "kind": "barcode", "value": "6009999999999"},
    )
    client.post(
        "/catalog/identifiers",
        json={"variant_id": "nb_milk_1l", "kind": "sku", "value": "NB-MILK-1L"},
    )
    by_barcode = client.get("/catalog/resolve?kind=barcode&value=6009999999999")
    by_sku = client.get("/catalog/resolve?kind=sku&value=NB-MILK-1L")
    assert product.status_code == 200 and variant.status_code == 200
    assert by_barcode.json()["variant"]["variant_id"] == by_sku.json()["variant"]["variant_id"]
    print("POST /catalog/* -> two different codes resolve to the SAME variant (identity proof)")

    systems = client.get("/connectors/systems")
    assert systems.status_code == 200
    print("GET /connectors/systems ->", len(systems.json()["systems"]), "connector systems")

    observability = client.get("/mlops/observability")
    assert observability.status_code == 200
    print("GET /mlops/observability -> tenant:", observability.json()["snapshot"]["tenant_id"])


run("In-process API smoke (TestClient)", _api_smoke)

## 5. Live HTTP server smoke

Boots the real ASGI app with `uvicorn` on an actual port in a background thread, waits for it to report healthy, hits it with real `httpx` requests over the network stack (not just in-process), then shuts it down cleanly. This is the closest thing to "run the app for real" without leaving a server hanging around after the notebook finishes.

In [ ]:
_LIVE_PORT = 8010
_live_server = None
_live_thread = None


def _live_server_smoke() -> None:
    global _live_server, _live_thread
    import asyncio
    import threading

    import httpx
    import uvicorn

    from shelfwise_backend.app import app as shelfwise_app

    config = uvicorn.Config(
        shelfwise_app, host="127.0.0.1", port=_LIVE_PORT, log_level="warning"
    )
    _live_server = uvicorn.Server(config)

    def _serve() -> None:
        asyncio.run(_live_server.serve())

    _live_thread = threading.Thread(target=_serve, daemon=True)
    _live_thread.start()

    base = f"http://127.0.0.1:{_LIVE_PORT}"
    deadline = time.monotonic() + 15
    last_error: Exception | None = None
    while time.monotonic() < deadline:
        try:
            response = httpx.get(f"{base}/health", timeout=1)
            if response.status_code == 200:
                break
        except Exception as exc:  # noqa: BLE001 - retry loop, report the last error on timeout
            last_error = exc
        time.sleep(0.2)
    else:
        raise RuntimeError(f"live server did not become healthy in time: {last_error}")

    health = httpx.get(f"{base}/health", timeout=5)
    assert health.status_code == 200, health.text
    print("live GET /health ->", health.json())

    golden = httpx.get(f"{base}/demo/golden", timeout=10)
    assert golden.status_code == 200, golden.text
    print("live GET /demo/golden -> decision:", golden.json()["decision"]["action"]["type"])


try:
    run("Live HTTP server smoke (uvicorn)", _live_server_smoke)
finally:
    if _live_server is not None:
        _live_server.should_exit = True
    if _live_thread is not None:
        _live_thread.join(timeout=5)
    print("live server stopped")

## 6. AMD MI300X / vLLM (or Fireworks) inference smoke — optional

The whole app runs deterministically **without** this (the default `offline` provider is what every check above just used). This cell is the one place you plug in the GPU/network credential — set these in the environment *before* starting the kernel, do not paste them into a cell:

- `LLM_BASE_URL` — the OpenAI-compatible endpoint (vLLM on this box, or Fireworks).
- `LLM_API_KEY` — the bearer token for that endpoint.
- `LLM_ROUTINE_MODEL` / `LLM_STRONG_MODEL` — optional, defaults to the served model names.

If these aren't set, this section reports **SKIPPED**, not failed — that's expected and does not mean anything is broken.

In [ ]:
def _inference_smoke() -> None:
    from shelfwise_inference import InferenceError, OpenAICompatibleInferenceClient, load_inference_config

    config = load_inference_config()
    print(json.dumps(config.to_public_dict(), indent=2))

    if not config.base_url or not config.api_key_present:
        print("SKIPPED: set LLM_BASE_URL and LLM_API_KEY in the environment to exercise this.")
        return

    client = OpenAICompatibleInferenceClient(config)
    started = time.monotonic()
    result = client.complete(
        agent="critic",
        system="You are the ShelfWise critic. Be concise and cite evidence gaps.",
        user=(
            "Review this recommendation: markdown SKU 4011 by 30 percent because expiry "
            "risk is high."
        ),
        max_tokens=120,
    )
    elapsed = time.monotonic() - started
    print(f"provider={result.provider} model={result.model} latency_ms={result.latency_ms} "
          f"wall_s={elapsed:.2f}")
    print("content:", result.content[:400])
    assert result.content.strip(), "model returned empty content"


run("AMD/vLLM inference smoke (optional)", _inference_smoke)

## 7. Docker Compose config validation — optional

Only meaningful if Docker is installed on this box. Validates `docker-compose.yml` (including the Postgres least-privilege role setup) without starting any containers.

In [ ]:
def _compose_check() -> None:
    result = subprocess.run(
        ["docker", "compose", "config", "--quiet"],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stderr[-2000:])
    assert result.returncode == 0, "docker compose config reported a problem"
    print("docker-compose.yml is valid")


try:
    run("Docker Compose config (optional)", _compose_check)
except Exception as exc:  # noqa: BLE001
    print(f"SKIPPED: Docker not available here ({exc})")

## Final summary

**Read this cell first when you come back to check results.**

In [ ]:
print("=" * 72)
print("SHELFWISE TEST HARNESS SUMMARY")
print("=" * 72)
width = max((len(item["section"]) for item in RESULTS), default=10)
failures = 0
for item in RESULTS:
    banner = "PASS" if item["ok"] else "FAIL"
    if not item["ok"]:
        failures += 1
    line = f"{banner:4}  {item['section']:<{width}}"
    if item["detail"]:
        line += f"  ({item['detail']})"
    print(line)
print("=" * 72)
if failures == 0:
    print("ALL CHECKS PASSED - the agent, the harness, and the API surface all work.")
else:
    print(f"{failures} check(s) FAILED - scroll up to the matching section for the full output.")